# N14: The Ceiling Breaker — TabPFN v3 + Numeric TargetEncoder + GBDT Diversity Ensemble**Objective:** Organically breach the 0.95011 barrier by combining three independently validated innovations that were never used together:1. **TabPFN v3** (Prior-Data Fitted Network, 2025-2026) — a pre-trained transformer that handles up to 1M rows natively. No bagging hack.2. **sklearn TargetEncoder on ALL features** (numeric + categorical) — Era 1's single biggest improvement (+0.0009 OOF), never ported to Era 2.3. **GBDT Ensemble with Numeric TE** (CatBoost + LightGBM + HGBC) — three tree families with the TE feature representation.4. **True Diversity Blend** — TabPFN (neural) x GBDT (tree) provides genuinely orthogonal error patterns for ensemble lift.**The mechanistic justification:** Era 2 hit 0.95011 using categorical-only TE. Era 1 reached 0.95065 using exact-value numeric TE + RealMLP. Neither era combined TabPFN v3 + numeric TE + multi-GBDT diversity. This notebook tests that untried combination.

In [ ]:
!pip install -q tabpfn tabm --no-deps 2>/dev/null || true!pip install -q tabpfn 2>/dev/null || true

In [ ]:
import pandas as pdimport numpy as npimport warningsimport timeimport torchfrom sklearn.model_selection import StratifiedKFoldfrom sklearn.metrics import balanced_accuracy_scorefrom sklearn.preprocessing import LabelEncoder, TargetEncoderfrom sklearn.impute import SimpleImputerfrom sklearn.ensemble import HistGradientBoostingClassifierfrom sklearn.utils.class_weight import compute_sample_weightimport lightgbm as lgbfrom catboost import CatBoostClassifierwarnings.filterwarnings('ignore')# Global ConfigID_COL, TARGET_COL = "id", "health_condition"CLASSES = ("at-risk", "fit", "unhealthy")N_FOLDS = 5SEED = 42N_CLASSES = 3# GPU Detection (Rule 12 Compliance)GPU_ENABLED = torch.cuda.is_available()DEVICE = 'cuda' if GPU_ENABLED else 'cpu'print(f"GPU Available: {GPU_ENABLED} | Device: {DEVICE}")
import os
# Prefer Kaggle Models-mounted TabPFN-3 weights (avoids PriorLabs browser license flow).
# Add model input: Models → prior-labsai/tabpfn-3 → Add to Notebook.
_KAGGLE_TABPFN = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
if os.path.isdir(_KAGGLE_TABPFN):
    os.environ["TABPFN_MODEL_CACHE_DIR"] = _KAGGLE_TABPFN
    print(f"Using mounted TabPFN weights: {_KAGGLE_TABPFN}")
elif not os.environ.get("TABPFN_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["TABPFN_TOKEN"] = UserSecretsClient().get_secret("TABPFN_TOKEN")
        print("Using TABPFN_TOKEN from Kaggle Secrets")
    except Exception:
        print(
            "WARNING: TabPFN-3 weights not mounted and TABPFN_TOKEN unset. "
            "Add prior-labsai/tabpfn-3 as a notebook input, or set Secret TABPFN_TOKEN."
        )

# TabPFN availability checkTABPFN_AVAILABLE = Falsetry:    
import os
os.environ["TABPFN_MODEL_CACHE_DIR"] = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
os.environ["TABPFN_NO_BROWSER"] = "1"
TABPFN_CKPT = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1/tabpfn-v3-classifier-v3_default.ckpt"
if not os.path.isfile(TABPFN_CKPT):
    raise FileNotFoundError(
        f"Missing mounted checkpoint: {TABPFN_CKPT}. "
        "Add Models → prior-labsai/tabpfn-3, then Factory reset + Run All."
    )
print(f"TabPFN checkpoint OK: {TABPFN_CKPT}")

from tabpfn import TabPFNClassifier    TABPFN_AVAILABLE = True    print("TabPFN v3 loaded successfully.")except ImportError:    print("TabPFN not available. Will use GBDT-only pipeline.")

In [ ]:
# 1. Load Data (Kaggle-first paths with local fallback)import oskaggle_paths = [    '/kaggle/input/competitions/playground-series-s6e7',    '/kaggle/input/playground-series-s6e7',]local_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'data') if not os.path.exists('/kaggle/input') else NoneDATA_DIR = Nonefor p in kaggle_paths:    if os.path.exists(p) and os.path.exists(os.path.join(p, 'train.csv')):        DATA_DIR = p        breakif DATA_DIR is None and local_path and os.path.exists(os.path.join(local_path, 'train.csv')):    DATA_DIR = local_pathif DATA_DIR is None:    DATA_DIR = '../../data'train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))print(f"Train: {train_df.shape} | Test: {test_df.shape}")print(f"Target distribution:\n{train_df[TARGET_COL].value_counts(normalize=True)}")le_target = LabelEncoder()y_train = le_target.fit_transform(train_df[TARGET_COL])cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality']num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']ALL_FEATURES = cat_cols + num_cols

In [ ]:
# 2. The Core Innovation: sklearn TargetEncoder on ALL features (numeric + categorical)# # Era 1's v0.7 used TargetEncoder(cv=5, target_type='multiclass') on ALL 13 features# (including numerics cast to string). This was the single biggest real improvement# in the entire project history (+0.0009 OOF over CatBoost baseline).## Era 2 NEVER used this. They used manual pd.crosstab on categoricals only.# Their one attempt at numeric TE "catastrophically collapsed" because they used# raw float crosstab (infinite cardinality), NOT sklearn's TargetEncoder which# handles this with proper smoothing and cross-fitting.def prepare_features_with_te(train_df, test_df, y_train, cat_cols, num_cols, fold_indices):    """Prepare features using sklearn TargetEncoder on ALL features (the v0.7 technique)."""    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)        # Prepare raw features: cast ALL to string for TargetEncoder    all_cols = cat_cols + num_cols    X_train_raw = train_df[all_cols].copy()    X_test_raw = test_df[all_cols].copy()        # Fill NaN for string conversion    for col in all_cols:        X_train_raw[col] = X_train_raw[col].fillna('__MISSING__').astype(str)        X_test_raw[col] = X_test_raw[col].fillna('__MISSING__').astype(str)        # Also prepare raw numeric features (imputed) for tree models    num_imputer = SimpleImputer(strategy='median')    X_train_num = pd.DataFrame(        num_imputer.fit_transform(train_df[num_cols]),        columns=num_cols, index=train_df.index    )    X_test_num = pd.DataFrame(        num_imputer.transform(test_df[num_cols]),        columns=num_cols, index=test_df.index    )        # Cross-fitted TargetEncoder on ALL features    n_train = len(X_train_raw)    n_test = len(X_test_raw)        # TargetEncoder produces N_CLASSES columns per input column for multiclass    te = TargetEncoder(cv=5, target_type='multiclass', random_state=SEED)        # Fit on full train, transform both    X_train_te = te.fit_transform(X_train_raw, y_train)    X_test_te = te.transform(X_test_raw)        # Combine: raw numerics + TE features    te_col_names = [f'te_{col}_c{c}' for col in all_cols for c in range(N_CLASSES)]        X_train_full = np.hstack([X_train_num.values, X_train_te])    X_test_full = np.hstack([X_test_num.values, X_test_te])        print(f"Feature matrix: {X_train_full.shape[1]} columns")    print(f"  Raw numeric: {len(num_cols)}")    print(f"  TE columns: {X_train_te.shape[1]} ({len(all_cols)} features x {N_CLASSES} classes)")        return X_train_full, X_test_full, X_train_raw, X_test_rawskf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)fold_indices = list(skf.split(train_df, y_train))X_train_full, X_test_full, X_train_raw, X_test_raw = prepare_features_with_te(    train_df, test_df, y_train, cat_cols, num_cols, fold_indices)

In [ ]:
# 3. ABLATION A: TabPFN v3 (Full Dataset, No Bagging)# The mechanistic justification: TabPFN v3 handles up to 1M rows natively.# N13 used the old TabPFN with 5K-row bagging — that's obsolete.oof_tabpfn = np.zeros((len(train_df), N_CLASSES))tst_tabpfn = np.zeros((len(test_df), N_CLASSES))tabpfn_scores = []if TABPFN_AVAILABLE:    print("=== ABLATION A: TabPFN v3 (Full Dataset) ===")    t0 = time.time()        for fold, (tr_i, va_i) in enumerate(fold_indices):        print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)                clf = TabPFNClassifier(device=DEVICE, model_path=TABPFN_CKPT)                try:            clf.fit(X_train_full[tr_i], y_train[tr_i])                        va_probs = clf.predict_proba(X_train_full[va_i])            te_probs = clf.predict_proba(X_test_full)                        oof_tabpfn[va_i] = va_probs            tst_tabpfn += te_probs / N_FOLDS                        fold_score = balanced_accuracy_score(y_train[va_i], va_probs.argmax(axis=1))            tabpfn_scores.append(fold_score)            print(f"BAcc={fold_score:.5f}")                    except Exception as e:            print(f"TabPFN failed on fold {fold+1}: {e}")            print("Falling back to subsample mode...")                        # Fallback: stratified subsample if full dataset too large for GPU memory            from sklearn.model_selection import train_test_split            max_rows = 100000            if len(tr_i) > max_rows:                sub_idx, _ = train_test_split(                    np.arange(len(tr_i)), train_size=max_rows,                    stratify=y_train[tr_i], random_state=SEED+fold                )                fit_idx = tr_i[sub_idx]            else:                fit_idx = tr_i                        clf = TabPFNClassifier(device=DEVICE, model_path=TABPFN_CKPT)            clf.fit(X_train_full[fit_idx], y_train[fit_idx])                        va_probs = clf.predict_proba(X_train_full[va_i])            te_probs = clf.predict_proba(X_test_full)                        oof_tabpfn[va_i] = va_probs            tst_tabpfn += te_probs / N_FOLDS                        fold_score = balanced_accuracy_score(y_train[va_i], va_probs.argmax(axis=1))            tabpfn_scores.append(fold_score)            print(f"  (subsample) BAcc={fold_score:.5f}")        tabpfn_oof_score = balanced_accuracy_score(y_train, oof_tabpfn.argmax(axis=1))    print(f"\n>>> TabPFN v3 OOF Balanced Accuracy: {tabpfn_oof_score:.5f} ({time.time()-t0:.0f}s) <<<")else:    print("TabPFN not available. Skipping Ablation A.")    tabpfn_oof_score = 0.0

In [ ]:
# 4. ABLATION B: GBDT Ensemble (CatBoost + LightGBM + HGBC) with Numeric TE# The mechanistic justification: This is the Era 1 v0.7 technique (exact-value# TargetEncoder on all features) applied to multiple tree architectures with # seed-blending for variance reduction (Era 2 N10's winning strategy).print("=== ABLATION B: GBDT Ensemble with Numeric TE ===")t0 = time.time()oof_gbdt = np.zeros((len(train_df), N_CLASSES))tst_gbdt = np.zeros((len(test_df), N_CLASSES))gbdt_scores = []GBDT_SEEDS = [42, 2026, 999]cb_task = "GPU" if GPU_ENABLED else "CPU"lgb_device = "gpu" if GPU_ENABLED else "cpu"for fold, (tr_i, va_i) in enumerate(fold_indices):    print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)        X_tr, X_va = X_train_full[tr_i], X_train_full[va_i]    X_te = X_test_full    y_tr, y_va = y_train[tr_i], y_train[va_i]        sample_weights = compute_sample_weight('balanced', y_tr)    class_counts = np.bincount(y_tr, minlength=N_CLASSES)    cb_weights = [len(y_tr) / (N_CLASSES * c) for c in class_counts]        fold_probs_va = np.zeros((len(va_i), N_CLASSES))    fold_probs_te = np.zeros((len(test_df), N_CLASSES))    n_models = 3 * len(GBDT_SEEDS)        for seed in GBDT_SEEDS:        # 1. CatBoost        m_cb = CatBoostClassifier(            iterations=1200, learning_rate=0.03, depth=6,            class_weights=cb_weights, random_seed=seed, verbose=0,            task_type=cb_task        )        m_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100)        fold_probs_va += m_cb.predict_proba(X_va) / n_models        fold_probs_te += m_cb.predict_proba(X_te) / n_models                # 2. LightGBM        lgb_params = dict(            n_estimators=1200, learning_rate=0.03, num_leaves=63,            class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1        )        if GPU_ENABLED:            lgb_params['device_type'] = 'gpu'        m_lgb = lgb.LGBMClassifier(**lgb_params)        m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],                  callbacks=[lgb.early_stopping(100, verbose=False)])        fold_probs_va += m_lgb.predict_proba(X_va) / n_models        fold_probs_te += m_lgb.predict_proba(X_te) / n_models                # 3. HistGradientBoosting        m_hgb = HistGradientBoostingClassifier(            max_iter=1200, learning_rate=0.03, max_leaf_nodes=63,            class_weight='balanced', random_state=seed,            early_stopping=True, validation_fraction=0.1, n_iter_no_change=100        )        m_hgb.fit(X_tr, y_tr, sample_weight=sample_weights)        fold_probs_va += m_hgb.predict_proba(X_va) / n_models        fold_probs_te += m_hgb.predict_proba(X_te) / n_models        oof_gbdt[va_i] = fold_probs_va    tst_gbdt += fold_probs_te / N_FOLDS        fold_score = balanced_accuracy_score(y_va, fold_probs_va.argmax(axis=1))    gbdt_scores.append(fold_score)    print(f"BAcc={fold_score:.5f}")gbdt_oof_score = balanced_accuracy_score(y_train, oof_gbdt.argmax(axis=1))print(f"\n>>> GBDT Ensemble (Numeric TE) OOF Balanced Accuracy: {gbdt_oof_score:.5f} ({time.time()-t0:.0f}s) <<<")

In [ ]:
# 5. True Diversity Blend: TabPFN x GBDT# The mechanistic justification: TabPFN (neural, in-context learning) and# GBDT (tree, gradient boosting) make genuinely orthogonal errors.# Era 1 showed RealMLP + HGBC-TE had a non-degenerate blend weight (86/14),# proving neural-tree diversity exists. TabPFN should be even more diverse.print("=== DIVERSITY BLEND OPTIMIZATION ===")if TABPFN_AVAILABLE and tabpfn_oof_score > 0.90:    # Grid search over blend weights    best_blend_score = 0    best_alpha = 0        for alpha in np.arange(0.0, 1.01, 0.02):        blend = alpha * oof_tabpfn + (1 - alpha) * oof_gbdt        score = balanced_accuracy_score(y_train, blend.argmax(axis=1))        if score > best_blend_score:            best_blend_score = score            best_alpha = alpha        print(f"TabPFN solo:  {tabpfn_oof_score:.5f}")    print(f"GBDT solo:    {gbdt_oof_score:.5f}")    print(f"Best blend:   {best_blend_score:.5f} (TabPFN={best_alpha:.2f}, GBDT={1-best_alpha:.2f})")        # Use best config for final predictions    if best_blend_score > max(tabpfn_oof_score, gbdt_oof_score):        final_probs = best_alpha * tst_tabpfn + (1 - best_alpha) * tst_gbdt        final_score = best_blend_score        final_method = f"Blend (TabPFN={best_alpha:.2f}, GBDT={1-best_alpha:.2f})"    elif tabpfn_oof_score > gbdt_oof_score:        final_probs = tst_tabpfn        final_score = tabpfn_oof_score        final_method = "TabPFN solo"    else:        final_probs = tst_gbdt        final_score = gbdt_oof_score        final_method = "GBDT Ensemble solo"else:    final_probs = tst_gbdt    final_score = gbdt_oof_score    final_method = "GBDT Ensemble solo (TabPFN unavailable)"print(f"\n>>> FINAL: {final_method} | OOF BAcc: {final_score:.5f} <<<")

In [ ]:
# 6. Optional: High-Confidence Pseudo-Labeling (N06 technique)# Only applied if the base score exceeds the current organic ceilingPSEUDO_THRESHOLD = 0.99APPLY_PSEUDO = True  # Set to False to skipif APPLY_PSEUDO:    print("=== PSEUDO-LABELING ENHANCEMENT ===")        max_probs = final_probs.max(axis=1)    pseudo_mask = max_probs >= PSEUDO_THRESHOLD    n_pseudo = pseudo_mask.sum()    print(f"Extracted {n_pseudo} pseudo-labels ({n_pseudo/len(test_df)*100:.1f}% of test)")        if n_pseudo > 1000:        pseudo_y = final_probs[pseudo_mask].argmax(axis=1)                # Augment training data        aug_X = np.vstack([X_train_full, X_test_full[pseudo_mask]])        aug_y = np.concatenate([y_train, pseudo_y])                # Retrain GBDT ensemble on augmented data        aug_tst_probs = np.zeros((len(test_df), N_CLASSES))        skf_aug = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED*2)        aug_scores = []                for fold, (tr_i, va_i) in enumerate(skf_aug.split(aug_X, aug_y)):            X_tr, X_va = aug_X[tr_i], aug_X[va_i]            y_tr, y_va = aug_y[tr_i], aug_y[va_i]                        sample_weights = compute_sample_weight('balanced', y_tr)            class_counts = np.bincount(y_tr, minlength=N_CLASSES)            cb_weights = [len(y_tr) / (N_CLASSES * c) for c in class_counts]                        fold_probs = np.zeros((len(test_df), N_CLASSES))            n_models = 3  # One seed for speed                        m_cb = CatBoostClassifier(                iterations=1200, learning_rate=0.03, depth=6,                class_weights=cb_weights, random_seed=SEED, verbose=0,                task_type=cb_task            )            m_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100)            fold_probs += m_cb.predict_proba(X_test_full) / n_models                        lgb_params = dict(                n_estimators=1200, learning_rate=0.03, num_leaves=63,                class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1            )            if GPU_ENABLED:                lgb_params['device_type'] = 'gpu'            m_lgb = lgb.LGBMClassifier(**lgb_params)            m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],                      callbacks=[lgb.early_stopping(100, verbose=False)])            fold_probs += m_lgb.predict_proba(X_test_full) / n_models                        m_hgb = HistGradientBoostingClassifier(                max_iter=1200, learning_rate=0.03, max_leaf_nodes=63,                class_weight='balanced', random_state=SEED,                early_stopping=True, validation_fraction=0.1, n_iter_no_change=100            )            m_hgb.fit(X_tr, y_tr, sample_weight=sample_weights)            fold_probs += m_hgb.predict_proba(X_test_full) / n_models                        aug_tst_probs += fold_probs / N_FOLDS            aug_score = balanced_accuracy_score(y_va, m_cb.predict(X_va))            aug_scores.append(aug_score)            print(f"  Aug Fold {fold+1}/{N_FOLDS} BAcc={aug_score:.5f}")                # Blend augmented predictions with base        # Use a conservative blend to avoid pseudo-label noise        final_probs_aug = 0.7 * final_probs + 0.3 * aug_tst_probs        final_method += " + PseudoLabel"        print(f"\nPseudo-labeling applied. Final blend: 70% base + 30% augmented")    else:        final_probs_aug = final_probs        print("Too few pseudo-labels. Skipping augmentation.")else:    final_probs_aug = final_probs

In [ ]:
# 7. Generate Submissionfinal_preds = final_probs_aug.argmax(axis=1)sub_df = pd.DataFrame({    ID_COL: test_df[ID_COL].astype("int64"),    TARGET_COL: le_target.inverse_transform(final_preds)})sub_df.to_csv("submission.csv", index=False)print(f"\n{'='*60}")print(f"N14: THE CEILING BREAKER — RESULTS")print(f"{'='*60}")if TABPFN_AVAILABLE and tabpfn_oof_score > 0.90:    print(f"TabPFN v3 solo OOF:     {tabpfn_oof_score:.5f}")print(f"GBDT Ensemble OOF:      {gbdt_oof_score:.5f}")print(f"Final method:           {final_method}")print(f"Final OOF BAcc:         {final_score:.5f}")print(f"{'='*60}")print(f"\nPrevious best organic:  0.95011 (N06)")print(f"Era 1 best organic:     0.95065 (v0.9)")print(f"Delta vs N06:           {final_score - 0.95011:+.5f}")print(f"Delta vs v0.9:          {final_score - 0.95065:+.5f}")print(f"\nSubmission saved to submission.csv ({len(sub_df)} rows)")print(sub_df[TARGET_COL].value_counts())

In [ ]:
# 8. Ablation Summary Tableprint("\n=== ABLATION SUMMARY ===")print(f"{'Method':<35} {'OOF BAcc':>10}")print(f"{'-'*45}")if TABPFN_AVAILABLE and tabpfn_oof_score > 0.90:    print(f"{'TabPFN v3 (full dataset)' :<35} {tabpfn_oof_score:>10.5f}")print(f"{'GBDT Ensemble (Numeric TE)' :<35} {gbdt_oof_score:>10.5f}")if TABPFN_AVAILABLE and tabpfn_oof_score > 0.90:    print(f"{'Best Blend' :<35} {best_blend_score:>10.5f}")print(f"{'-'*45}")print(f"\nComparison to prior work:")print(f"  N04 (Cat TE only):     0.94955")print(f"  N06 (PseudoLabel):     0.95011")print(f"  v0.7 (HGBC+NumericTE): 0.95020")print(f"  v0.8 (RealMLP):        0.95062")print(f"  v0.9 (RealMLP+HGBC):   0.95065")print(f"  N14 (THIS):            {final_score:.5f}")